In [2]:
import torch
import numpy as np

a = torch.tensor([np.zeros(768) for i in range(35)])
a.shape

C:\Users\t1234\AppData\Local\Temp\ipykernel_21980\1690915949.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  a = torch.tensor([np.zeros(768) for i in range(35)])


torch.Size([35, 768])

In [18]:
a = [torch.tensor([1, 2]), torch.tensor([3, 4])]
torch.stack(a)

tensor([[1, 2],
        [3, 4]])

In [7]:
class NCMLoss(nn.Module):
    def __init__(self, temperature=1.0, epsilon=1e-8):
        """
        Args:
            temperature (float): 温度系数，用于缩放距离影响（类似对比学习）
            epsilon (float): 数值稳定项，防止除零错误
        """
        super().__init__()
        self.temperature = temperature
        self.epsilon = epsilon

    def forward(self, features, labels, prototypes=None):
        """
        Args:
            features: 输入特征 [B, D]
            labels: 真实标签 [B]
            prototypes: 可选，预定义类别原型矩阵 [C, D]

        Returns:
            loss: NCM分类损失
        """
        # 动态计算原型（若未提供）
        if prototypes is None:
            prototypes = self._compute_prototypes(features, labels)  # [C_batch, D]

        # 计算特征与所有原型的距离 [B, C]
        distances = torch.cdist(features, prototypes, p=2)  # 欧氏距离

        # 将距离转换为概率（距离越小概率越高）
        logits = -distances / self.temperature  # [B, C]

        # 计算交叉熵损失（需对齐原型索引与标签）
        unique_labels = torch.unique(labels)
        label_mapping = {l.item(): idx for idx, l in enumerate(unique_labels)}
        mapped_labels = torch.tensor([label_mapping[l.item()] for l in labels],
                                   device=features.device)

        loss = F.cross_entropy(logits, mapped_labels)
        return loss

    def _compute_prototypes(self, features, labels):
        """
        根据当前批次动态计算每个类别的原型（均值）
        Args:
            features: [B, D]
            labels: [B]
        Returns:
            prototypes: [C_batch, D]
        """
        unique_labels = torch.unique(labels)
        prototypes = []
        for c in unique_labels:
            mask = (labels == c)
            if mask.sum() == 0:  # 防御：跳过空类别
                continue
            class_features = features[mask]
            proto = class_features.mean(dim=0)  # [D]
            prototypes.append(proto)

        # 处理全空情况（理论上不会发生）
        if len(prototypes) == 0:
            return torch.randn(1, features.size(1), device=features.device)

        return torch.stack(prototypes, dim=0)  # [C_batch, D]

    def extra_repr(self):
        return f"temperature={self.temperature}, epsilon={self.epsilon}"

4